# Grant Award Categorisation — INFERENCE (production)
Runs the **frozen** categorisation classifier over all TRUE grants from the false-positive step, and writes predictions to merge back **before `05_enrich`**.

This is the production sibling of the eval bench. Prompt, few-shot, and affiliation rules are frozen — **do not tune here.** If you need to change behaviour, do it in the eval bench, re-validate against the 80-row gold, then copy the frozen prompt back here.

**Design (recorded):** affiliation matching is prompt-based (LLM matches recipient to injected trigger-orgs); no ROR resolution. See the eval notebook's decision cell for the full rationale + known limitation (silent match misses).

**Key handling:** `award_id` is unusable (44 blanks). This notebook validates `ID_COL` (non-null + unique) and also stamps a positional `infer_idx`, so merge-back has a reliable join regardless.

---

**Model (frozen):** Claude Haiku (`claude-haiku-4-5-20251001`). Validated across 3 runs each vs Sonnet on the 80-award gold: statistically indistinguishable (Cohen's κ ≈ 0.74–0.78 vs human ceiling 0.59), identical on the DIRECT/ADJACENT boundary that drives the funding split. Haiku chosen on cost (~10× cheaper). Both under-detect ADOPT/UNKNOWN — those figures are indicative.

## 1. Setup

In [1]:
!pip install anthropic pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 6.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import anthropic, pandas as pd, numpy as np, time, json, os, ast, re
from google.colab import userdata
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
print("\u2713 setup")

✓ setup


## 2. Configuration

In [8]:
# \u2500\u2500 EDIT \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
BASE = '/content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding'
INPUT_PATH = f'{BASE}/deduplicated_clean.csv'   # FP-passed TRUE grants
GOLD_PATH  = f'{BASE}/award type classifier/award_type_classifer_gold.csv'             # only used to pull few-shot examples
AFFIL_PATH = f'{BASE}/award type classifier/oi_affiliations_complete.csv'
OUTPUT_PATH= f'{BASE}/categorised_predictions.csv'
CKPT       = f'{BASE}/categorised_infer_ckpt.csv'

MODEL = 'claude-haiku-4-5-20251001'  # FROZEN production choice (see note below)
# MODEL = 'claude-sonnet-4-6'        # alt; indistinguishable on gold, ~10x cost

# Stable id for join + checkpoint. VALIDATED below - if it fails, notebook falls back to infer_idx.
ID_COL = 'record_uid'

# Column names in INPUT_PATH (edit to match deduplicated_clean.csv)
OI_COL='OI'; HOST_COL='host_organization'; RECIP_COL='RECIPIENT'
TITLE_COL='TITLE'; DESC_COL='DESCRIPTION'
# HOST_COL may be absent in the pipeline file - that's fine, affiliations still supply host/trigger orgs.

FEWSHOT_ROW_NOS = [1, 7, 9, 14, 28, 29]
AFFIL_OI_COL='oi_name'; AFFIL_TRIGGER_COL='triggers_direct_orgs'
API_DELAY=0.1; CKPT_EVERY=50
# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500

SUPERCATS=['DIRECT','ADJACENT','ADOPT','UNKNOWN']
DIRECT_ACT=['RESEARCH & DEVELOPMENT','OPERATIONS','OTHER']
def leaf_to_super(l): return 'DIRECT' if l in DIRECT_ACT else l
print("model:",MODEL)

model: claude-haiku-4-5-20251001


## 3. Affiliations lookup + few-shot (frozen)

In [5]:
def parse_list(x):
    if pd.isna(x): return []
    s=str(x).strip()
    for L in (json.loads, ast.literal_eval):
        try:
            v=L(s); return [str(i).strip() for i in v] if isinstance(v,list) else [str(v).strip()]
        except: pass
    return [s]
def norm(x):
    if pd.isna(x): return np.nan
    s=re.sub(r'\s+',' ',str(x).strip().upper())
    return {'R&D':'RESEARCH & DEVELOPMENT','RESEARCH AND DEVELOPMENT':'RESEARCH & DEVELOPMENT',
            'OPS':'OPERATIONS','OTHERS':'OTHER','ADOPTION':'ADOPT'}.get(s,s)
def norm_oi(s): return re.sub(r'[^a-z0-9]+','',str(s).lower())

aff=pd.read_csv(AFFIL_PATH, dtype=str).fillna('')
def clean_orgs(cell):
    out=[]
    for o in str(cell).split(';'):
        o=re.sub(r'\([^)]*\)','',o).strip()
        if o and o.lower() not in ('none','n/a'): out.append(o)
    return out
AFF_LOOKUP={ norm_oi(r[AFFIL_OI_COL]): clean_orgs(r[AFFIL_TRIGGER_COL]) for _,r in aff.iterrows() }
def triggers_for(oi_list):
    out=[]
    for oi in oi_list:
        k=norm_oi(oi)
        if k in AFF_LOOKUP: out+=AFF_LOOKUP[k]; continue
        for ak,orgs in AFF_LOOKUP.items():
            if ak and (ak in k or k in ak): out+=orgs; break
    seen=set(); res=[]
    for o in out:
        if o.lower() not in seen: seen.add(o.lower()); res.append(o)
    return res

gold=pd.read_csv(GOLD_PATH, quoting=1)
gold['award_supercat']=gold['award_supercat'].map(norm); gold['award_cat']=gold['award_cat'].map(norm)
def render_input(oi_field, host_field, recipient, title, desc):
    ois=parse_list(oi_field); hosts=parse_list(host_field) if host_field is not None else []
    pairs=[]
    for i,o in enumerate(ois):
        h=hosts[i] if i<len(hosts) and str(hosts[i]) not in ('','nan','None') else None
        pairs.append(f"{o} (host: {h})" if h else o)
    trig=triggers_for(ois)
    lines=[f"OI(s): {'; '.join(pairs) if pairs else 'n/a'}",
           f"Recipient: {recipient if pd.notna(recipient) else 'not available'}"]
    if trig: lines.append("Known DIRECT-trigger orgs for these OI(s): " + "; ".join(trig))
    lines += [f"Title: {title}", f"Description: {desc if pd.notna(desc) else ''}"]
    return "\n".join(lines)

FEWSHOT_MSGS=[]
for _,r in gold[gold['row_no'].isin(FEWSHOT_ROW_NOS)].iterrows():
    inp=render_input(r['oi_name'],r.get('host_organization'),r.get('recipient'),r['grant_title'],r.get('grant_description'))
    sup=r['award_supercat']; leaf=r['award_cat']
    FEWSHOT_MSGS += [{"role":"user","content":inp},
                     {"role":"assistant","content":json.dumps({"reasoning":"(gold example)","supercat":sup,"activity": leaf if sup=='DIRECT' else None})}]
print("affiliation OIs:",len(AFF_LOOKUP),"| few-shot messages:",len(FEWSHOT_MSGS))

affiliation OIs: 135 | few-shot messages: 12


## 4. Frozen prompt + classify

In [6]:
SYSTEM_PROMPT = """You categorise a research grant by its relationship to the open infrastructure(s) (OI) it involves. OI involvement is already confirmed upstream \u2014 do not question it.

Apply this decision procedure IN ORDER, stopping at the first match:

1) Does the grant's recipient match the OI's HOST organisation, OR any org in "Known DIRECT-trigger orgs" for this grant (current/former hosts, founders, fiscal conduits)? Recipient and org may be worded differently but refer to the same organisation.
   \u2192 DIRECT. Then choose the activity:
      \u2022 RESEARCH & DEVELOPMENT \u2014 R&D incl. software development (may be by an org other than the host).
      \u2022 OPERATIONS \u2014 basic operations, maintenance/updates not counted as new development.
      \u2022 OTHER \u2014 other direct support, or multiple activities with no clear primary one.

2) Otherwise, is the grant's DELIVERABLE the uptake of the OI itself \u2014 a community/institution deploying it, migrating onto it, standing up a new instance for others, or driving sector-wide adoption? The OI's expanded use IS the goal.
   \u2192 ADOPT.

3) Otherwise, is the OI a TOOL toward a different deliverable (research output, digitised collection, software)? Depositing/uploading into it, ingesting into it, or building a plug-in/tool FOR it all count here.
   \u2192 ADJACENT.

4) Otherwise, OI is genuinely involved but you cannot place the relationship from the information given?
   \u2192 UNKNOWN.

Rules:
\u2022 Rule 1 is a BRIGHT LINE: recipient = host/former-host/conduit \u2192 DIRECT regardless of what the money funds.
\u2022 UNKNOWN = "can't place the relationship", NOT "I doubt the OI is involved".
\u2022 Reason briefly, then output ONLY this JSON (no markdown):
{"reasoning":"<one sentence>","supercat":"<DIRECT|ADJACENT|ADOPT|UNKNOWN>","activity":"<RESEARCH & DEVELOPMENT|OPERATIONS|OTHER|null>"}
activity is null unless supercat is DIRECT."""

def classify(oi_field, host_field, recipient, title, desc, max_retries=2):
    msgs=FEWSHOT_MSGS+[{"role":"user","content":render_input(oi_field,host_field,recipient,title,desc)}]
    for att in range(max_retries+1):
        try:
            resp=client.messages.create(model=MODEL,max_tokens=350,system=SYSTEM_PROMPT,messages=msgs)
            obj=json.loads(resp.content[0].text.strip().replace("```json","").replace("```","").strip())
            sup=str(obj.get("supercat","")).strip().upper()
            act=obj.get("activity",None); act=str(act).strip().upper() if act not in (None,"null","NULL","") else None
            if sup not in SUPERCATS: raise ValueError("bad supercat "+sup)
            if sup=='DIRECT':
                if act not in DIRECT_ACT: raise ValueError("DIRECT needs activity")
                leaf=act
            else: act=None; leaf=sup
            return {"leaf":leaf,"supercat":sup,"activity":act,"reasoning":str(obj.get("reasoning","")),"ok":True}
        except Exception as e:
            if att<max_retries: time.sleep(1.5); continue
            return {"leaf":"PARSE_ERROR","supercat":"PARSE_ERROR","activity":None,"reasoning":f"{type(e).__name__}: {e}","ok":False}
print("prompt frozen.")

prompt frozen.


## 5. Load input + VALIDATE THE JOIN KEY
Fails loudly if `ID_COL` isn't safe, and always stamps a positional `infer_idx` fallback.

In [9]:
df=pd.read_csv(INPUT_PATH, quoting=1)
df=df.reset_index(drop=True)
df['infer_idx']=range(len(df))          # positional fallback key, always safe within a single run

print(f"Input rows: {len(df)}")
print("Columns:", df.columns.tolist())

key_ok=True
if ID_COL not in df.columns:
    print(f"\u26a0 ID_COL '{ID_COL}' NOT in input \u2014 will use infer_idx for join/checkpoint."); key_ok=False
else:
    n_null=df[ID_COL].isna().sum(); n_dup=df[ID_COL].duplicated().sum()
    print(f"{ID_COL}: nulls={n_null}, duplicates={n_dup}")
    if n_null or n_dup:
        print(f"\u26a0 '{ID_COL}' is NOT a safe key (nulls/dups). Using infer_idx instead. "
              f"For merge-back you MUST join on infer_idx AND preserve input row order, "
              f"or fix {ID_COL} upstream."); key_ok=False
    else:
        print(f"\u2713 '{ID_COL}' is unique + non-null \u2014 safe join key.")

JOIN_KEY = ID_COL if key_ok else 'infer_idx'
print("JOIN_KEY =", JOIN_KEY)

# sanity: required text columns present
for c in [OI_COL, RECIP_COL, TITLE_COL]:
    if c not in df.columns: print(f"\u26a0 expected column '{c}' missing \u2014 edit the *_COL config.")
print("HOST_COL present:", HOST_COL in df.columns, "(optional - affiliations supply hosts if absent)")

Input rows: 1175
Columns: ['source_dataset', 'source_grantid', 'record_uid', 'FUNDER', 'RECIPIENT', 'RECIPIENT_LOCATION', 'OI', 'PI_NAME', 'START_DATE', 'END_DATE', 'GRANT_YEAR', 'GRANT_DURATION', 'AMOUNT', 'CURRENCY', 'SOURCE', 'SOURCE_URL', 'GRANT_ID', 'TITLE', 'DESCRIPTION', 'FUNDER_PROGRAM', 'award_cluster_id', 'dedup_status', 'predicted', 'reasoning', 'infer_idx']
record_uid: nulls=0, duplicates=0
✓ 'record_uid' is unique + non-null — safe join key.
JOIN_KEY = record_uid
HOST_COL present: False (optional - affiliations supply hosts if absent)


## 6. Run inference (checkpointed on JOIN_KEY)

In [ ]:
done={}
if os.path.exists(CKPT):
    prev=pd.read_csv(CKPT); done={str(r[JOIN_KEY]):r for _,r in prev.iterrows()}
    print(f"resume: {len(done)} already done")

out=[]
for i,r in df.iterrows():
    kid=str(r[JOIN_KEY])
    if kid in done: out.append(done[kid]); continue
    host_val=r[HOST_COL] if HOST_COL in df.columns else None
    res=classify(r[OI_COL], host_val, r.get(RECIP_COL), r.get(TITLE_COL,''), r.get(DESC_COL,''))
    rec={JOIN_KEY:r[JOIN_KEY],'infer_idx':r['infer_idx'],
         'pred_cat':res['leaf'],'pred_supercat':res['supercat'],'pred_activity':res['activity'],
         'reasoning':res['reasoning'],'ok':res['ok']}
    if ID_COL in df.columns: rec[ID_COL]=r[ID_COL]
    out.append(rec)
    if (i+1)%CKPT_EVERY==0: pd.DataFrame(out).to_csv(CKPT,index=False); print(f"  {i+1}/{len(df)}")
    time.sleep(API_DELAY)

pred=pd.DataFrame(out)
# derive supercat deterministically from leaf as a consistency guard
pred['pred_supercat']=pred['pred_cat'].map(lambda l: leaf_to_super(l) if l in (DIRECT_ACT+SUPERCATS) else pred['pred_supercat'])
pred.to_csv(CKPT,index=False)
print("done. rows:",len(pred),"| parse errors:",(~pred['ok']).sum())

  50/1175
  100/1175
  150/1175
  200/1175
  250/1175
  300/1175
  350/1175
  400/1175
  450/1175
  500/1175
  550/1175
  600/1175
  650/1175
  700/1175
  750/1175
  800/1175
  850/1175
  900/1175
  950/1175
  1000/1175
  1050/1175
  1100/1175
  1150/1175
done. rows: 1175 | parse errors: 0


## 7. Write predictions for merge-back

In [ ]:
pred.to_csv(OUTPUT_PATH, index=False)
print("Wrote", OUTPUT_PATH)
print("\nPredicted super-category distribution (count):")
print(pred['pred_supercat'].value_counts().to_string())
n_err=(~pred['ok']).sum()
if n_err: print(f"\n\u26a0 {n_err} PARSE_ERROR rows \u2014 inspect/re-run before merging:");
print("\nMerge back into the pipeline on:", JOIN_KEY, "(before 03_enrich).")
print("Reminder: dollar maths, funder concentration, and the funding split all happen DOWNSTREAM on the enriched data \u2014 not here.")